# Overview

This notebook demonstrates how to scan for TF binding motifs. The base GRN will be generated by combining the ATAC-seq peaks and motif information.

### Notebook file
Notebook file is available on CellOracle's GitHub page.
https://github.com/morris-lab/CellOracle/blob/master/docs/notebooks/02_motif_scan/02_atac_peaks_to_TFinfo_with_celloracle_20200801.ipynb


# 0. Import libraries

In [1]:
# Run before scan
from gimmemotifs.config import MotifConfig
# Single-thread BLAS to avoid cluster deadlock
MotifConfig().set_default_params({"ncpus": 1})

/cluster2/huanglab/jiamao/conda/envs/celloracle/lib/python3.10/site-packages/gimmemotifs/config.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm

In [3]:
import celloracle as co
from celloracle import motif_analysis as ma
from celloracle.utility import save_as_pickled_object
co.__version__

which: no R in (/cluster2/huanglab/jiamao/conda/envs/celloracle/bin:/cluster/home/jiamao/.cursor-server/bin/b3573281c4775bfc6bba466bf6563d3d498d1070/bin/remote-cli:/opt/simplehpc/scm/bin:/opt/simplehpc/scm/bin:/cluster/home/jiamao/.cargo/bin:/opt/simplehpc/scm/bin:/opt/simplehpc/scm/bin:/cluster2/huanglab/jiamao/conda/envs/celloracle/bin:/cluster2/huanglab/jiamao/conda/condabin:/opt/simplehpc/scm/bin:/usr/share/Modules/bin:/usr/local/bin:/usr/bin:/usr/local/sbin:/usr/sbin:/cluster/home/jiamao/.local/bin:/cluster/home/jiamao/bin:/cluster2/huanglab/jiamao/Apps/HOMER/bin:/cluster2/huanglab/jiamao/Apps/liftOver:/cluster/home/jiamao/.local/bin:/cluster/home/jiamao/bin:/cluster2/huanglab/jiamao/Apps/HOMER/bin:/cluster2/huanglab/jiamao/Apps/liftOver)


'0.20.0'

In [4]:
%config InlineBackend.figure_format = 'retina'
%matplotlib inline

plt.rcParams['figure.figsize'] = (15,7)
plt.rcParams["savefig.dpi"] = 600

# Gernerate GRN from Scenicplus

In [ ]:
# tf_to_gene_adj.tsv from Scenicplus
tf_to_gene_adj = pd.read_csv("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Oligo/outs/tf_to_gene_adj.tsv", sep="\t")
tf_to_gene_adj.shape

(2404491, 5)

In [ ]:
# region_to_gene_adj.tsv from Scenicplus
region_to_gene_adj = pd.read_csv("/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Scenicplus/Oligo/outs/region_to_gene_adj.tsv", sep="\t")
region_to_gene_adj.shape

(422371, 7)

In [41]:
import pandas as pd
import ast 

region_df = region_to_gene_adj.copy()
# prepare the distance column
def parse_distance(x):
    if isinstance(x, str):
        try:
            return abs(ast.literal_eval(x)[0])
        except:
            return 999999999
    elif isinstance(x, list):
        return abs(x[0])
    return abs(x)

region_df['abs_distance'] = region_df['Distance'].apply(parse_distance)

# set the filtering threshold
# condition 1: must be positively correlated (Open chromatin -> Gene Expression)
# set a slightly higher threshold 0.03, filter out very weak correlations
cond_rho = region_df['rho'] > 0.03 

# condition 2: Importance must be high
# strategy: take the 50th percentile of Importance (keep top 50% strong connections)
importance_cutoff = region_df['importance'].quantile(0.5)
cond_imp = region_df['importance'] > importance_cutoff

# condition 3: distance should not be too far
cond_dist = region_df['abs_distance'] < 500000


# filtering
region_filtered = region_df[cond_rho & cond_imp & cond_dist].copy()
print(f"original number of connections: {len(region_df)}")
print(f"number of connections after filtering: {len(region_filtered)}")

# preview
region_filtered.head()

original number of connections: 422371
number of connections after filtering: 61474


,target,region,importance,rho,importance_x_rho,importance_x_abs_rho,Distance,abs_distance
14,A2ML1,chr12:8689692-8690193,0.084664,0.069975,0.005924,0.005924,[132679],132679
16,A2ML1,chr12:8687707-8688208,0.104774,0.066157,0.006932,0.006932,[134663],134663
17,A2ML1,chr12:8697641-8698142,0.052208,0.036050,0.001882,0.001882,[124729],124729
21,A2ML1,chr12:8914190-8914691,0.044473,0.036853,0.001639,0.001639,[-27439],27439
23,A2ML1,chr12:8698402-8698903,0.059290,0.042637,0.002528,0.002528,[123969],123969


In [44]:
tf_filtered = tf_to_gene_adj[tf_to_gene_adj['regulation'].isin([1, -1])].copy()
region_df = region_filtered.copy()

# Merge the region and TF information
# this will produce a long table: Region - Gene - TF
merged_df = pd.merge(
    region_df[['region', 'target']],  # left table: Region and Gene
    tf_filtered[['TF', 'target']],    # right table: TF and Gene
    on='target',                      # join key
    how='inner'                       # take the intersection (i.e., the gene must have both Region and TF)
)

# fill the value with 1 (connection)
merged_df['value'] = 1 

# fill the missing value with 0
oligo_GRN = merged_df.pivot_table(
    index=['region', 'target'], 
    columns='TF', 
    values='value', 
    fill_value=0
)

# format adjustment (adapt to CellOracle format)
oligo_GRN = oligo_GRN.reset_index()


# gene_short_name corresponds to target
oligo_GRN = oligo_GRN.rename(columns={
    'region': 'peak_id',
    'target': 'gene_short_name'
})

# check the result
print(f"generated matrix dimension: {oligo_GRN.shape}")

# ensure the data type is float (CellOracle sometimes requires float)
oligo_GRN.iloc[:, 2:] = oligo_GRN.iloc[:, 2:].astype(float)
oligo_GRN.head()

generated matrix dimension: (61474, 705)


TF,peak_id,gene_short_name,ACAA1,ADNP2,AHCTF1,ALX1,ALX3,ALX4,AR,ARID3A,...,ZNF860,ZNF875,ZNF880,ZNF90,ZNF91,ZNF93,ZNF98,ZSCAN22,ZSCAN26,ZSCAN29
0,chr10:100009054-100009555,DNMBP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,chr10:100020821-100021322,DNMBP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,chr10:100025575-100026076,DNMBP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,chr10:100073098-100073599,CHUK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,chr10:100246930-100247431,CHUK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [48]:
oligo_GRN.columns.name = None

In [49]:
oligo_GRN.to_parquet("oligo_GRN_dataframe.parquet")